# Phase 2a: Accuracy reproduction on GPT-J 6B

**Goal.** Reproduce KT 2025's reported 80.5% accuracy on two-digit addition with GPT-J 6B, on the Phase-1 intersection set. Establishes that the prompt + tokenizer + greedy-decode pipeline is set up correctly; pre-registered gate before activation extraction.

**Runtime.** A100 40GB recommended (also works on T4 16GB; ~3x slower). Estimated wall time on A100: 5-15 minutes (model download dominates).

**Prompt.** `"Output ONLY a number. {a}+{b}="` (KT 2025 Table 2).

**Inputs from Drive.**
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/intersection.json` (Phase 1 output).

**Outputs to Drive.**
- `/MyDrive/blackbox_nlp_2026/correctness/gpt-j-6b.parquet` (cols: a, b, s, predicted_id, predicted_str, expected_id, correct).

**Pre-registered sanity gate.** `|empirical accuracy - 80.5%| <= 5pp`. If fail, halt and re-check the prompt template before Phase 3.

## 1. Standard prelude

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
PROJECT = '/content/drive/MyDrive/blackbox_nlp_2026'
CODE_DIR = f'{PROJECT}/code_repo'
REPO_URL = 'https://github.com/anshulk-cmu/blackbox-nlp-2026.git'

os.makedirs(PROJECT, exist_ok=True)
if not os.path.exists(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}
!git pull --ff-only

In [ ]:
!pip install -q \
    transformers==4.45.0 \
    huggingface_hub==0.25.1 \
    pandas==2.2.2 \
    pyarrow==17.0.0 \
    accelerate==1.0.1

In [ ]:
from google.colab import userdata
import huggingface_hub
hf_token = userdata.get('HF_TOKEN')
assert hf_token is not None and hf_token.startswith('hf_'), 'HF_TOKEN not set in Colab Secrets.'
huggingface_hub.login(hf_token)
print('HF login OK')

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU required for Phase 2. Switch runtime: Runtime > Change runtime type > A100/T4.'
print(f'CUDA: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load model + tokenizer

In [ ]:
sys.path.insert(0, f'{CODE_DIR}/code')
from accuracy_check import (
    MODEL_CONFIG, load_model_and_tokenizer, run_accuracy_check,
    save_correctness, summary_report, passes_sanity_gate,
    load_intersection_pairs,
)

MODEL_KEY = 'gpt-j-6b'
print(f'Loading {MODEL_KEY} ({MODEL_CONFIG[MODEL_KEY]["hf_name"]})...')
model, tokenizer = load_model_and_tokenizer(MODEL_KEY, device='cuda')
print(f'  dtype: {next(model.parameters()).dtype}')
print(f'  device: {next(model.parameters()).device}')
print(f'  param count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')

## 3. Load Phase-1 intersection

In [ ]:
intersection_path = f'{PROJECT}/tokenizer_audit/intersection.json'
assert os.path.exists(intersection_path), \
    f'{intersection_path} missing. Run Phase 1 (notebooks/01_tokenizer_audit.ipynb) first.'
pairs = load_intersection_pairs(intersection_path)
print(f'Loaded {len(pairs)} intersection pairs from Phase 1.')

## 4. Run greedy-decode accuracy check

Forward-passes each prompt once and takes argmax of the last-position next-token logits. With left-padding (set in `load_model_and_tokenizer`), the next-token position is always row index `-1`, so no per-row position bookkeeping is needed.

Saves a partial parquet to Drive every 200 batches; if the session drops, re-running this cell resumes from the partial.

In [ ]:
OUT_DIR = f'{PROJECT}/correctness'
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = f'{OUT_DIR}/{MODEL_KEY}.parquet'

df = run_accuracy_check(
    model, tokenizer, pairs, MODEL_KEY,
    batch_size=32,
    progress_every=10,
    save_every=200,
    partial_path=OUT_PATH,
)
save_correctness(df, OUT_PATH)
print(f'\nSaved {len(df)} rows to {OUT_PATH}')

## 5. Summary report and sanity gate

In [ ]:
print(summary_report(df, MODEL_KEY))
print()
if passes_sanity_gate(df, MODEL_KEY, tolerance_pp=5.0):
    print('PASS: empirical accuracy within 5pp of KT. Proceed to Phase 3.')
else:
    print('FAIL: empirical accuracy more than 5pp from KT. Halt and re-check.')
    print('Common causes: wrong prompt template, wrong dtype, tokenizer pad_token issue.')

## 6. Cleanup (free GPU memory before next phase)

In [ ]:
del model
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'Freed. CUDA memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')